## Notebook 1 — House Price Prediction (Linear Output + MSE)

### Project overview
- What is regression?
- Why this problem uses a Linear output layer.
- Why we use Mean Squared Error (MSE).

In [5]:
# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [7]:
!pip install datasets

   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 555.1/555.1 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/765.1 kB ? eta -:--:--
   --------------------------- ------------ 524.3/765.1 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 765.1/765.1 kB 2.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   --------------- ------------------------ 1.6/4.0 MB 7.2 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 9.3 MB/s  0:00:00
   ---------------------------------------- 0.0/27.3 MB ? eta -:--:--
   --- ------------------------------------ 2.1/27.3 MB 37.6 MB/s eta 0:00:01
   -------- ------------------------------- 6.0/27.3 MB 14.4 MB/s eta 0:00:02
   --------------- ------------------------ 10.7/27.3 MB 16.8 MB/s eta 0:00:01
   -------------------- ------

In [14]:
# CELL 2 — Load dataset (Hugging Face SMS Spam Collection)
# pip install datasets  (run once in terminal or !pip install datasets)
from datasets import load_dataset
import pandas as pd

# Use the updated 'namespace/name' format to avoid HfUriError
ds = load_dataset("ucirvine/sms_spam")

df = pd.DataFrame(ds["train"])
df.columns = ["text", "label"]  # label: 0 = ham, 1 = spam
df.head()

c:\Users\chari\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chari\.cache\huggingface\hub\datasets--ucirvine--sms_spam. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 5574/5574 [00:00<00:00, 144532.82 examples/s]


,text,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [15]:
# CELL 3 — Feature engineering (length, links, capitals, word freq proxy)
def count_links(text):
    return text.lower().count("http") + text.lower().count("www")

def count_capitals(text):
    return sum(1 for c in text if c.isupper())

def word_count(text):
    return len(text.split())

df["length"] = df["text"].apply(len)
df["num_links"] = df["text"].apply(count_links)
df["num_capitals"] = df["text"].apply(count_capitals)
df["num_words"] = df["text"].apply(word_count)

# Simple spam-keyword frequency feature
spam_keywords = ["free", "win", "winner", "cash", "prize", "urgent", "click", "offer"]
def keyword_freq(text):
    text = text.lower()
    return sum(text.count(k) for k in spam_keywords)

df["keyword_freq"] = df["text"].apply(keyword_freq)

feature_cols = ["length", "num_links", "num_capitals", "num_words", "keyword_freq"]
X = df[feature_cols].values.astype(np.float32)
y = df["label"].values.astype(np.float32)

df[feature_cols + ["label"]].describe()

,length,num_links,num_capitals,num_words,keyword_freq,label
count,5574.000000,5574.000000,5574.000000,5574.000000,5574.000000,5574.000000
mean,81.478292,0.022067,5.631324,15.591676,0.154826,0.134015
std,59.848302,0.162015,11.679841,11.390454,0.515224,0.340699
min,3.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,37.000000,0.000000,1.000000,7.000000,0.000000,0.000000
50%,63.000000,0.000000,2.000000,12.000000,0.000000,0.000000
75%,123.000000,0.000000,4.000000,23.000000,0.000000,0.000000
max,911.000000,2.000000,129.000000,171.000000,5.000000,1.000000


In [16]:
# CELL 4 — Train/test split, scaling, tensors
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

print(X_train_t.shape, y_train_t.shape)

torch.Size([4459, 5]) torch.Size([4459, 1])


In [17]:
# CELL 5 — Model definition
class SpamClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # raw logit — sigmoid applied via loss function
        )

    def forward(self, inputs):
        return self.net(inputs)
            
model = SpamClassifier(input_dim=X_train_t.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()  # sigmoid + BCE combined, numerically stable
optimizer = optim.Adam(model.parameters(), lr=0.01)
print(model)

SpamClassifier(
  (net): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [18]:
# CELL 6 — Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} — Loss: {loss.item():.4f}")

Epoch 10/100 — Loss: 0.5970
Epoch 20/100 — Loss: 0.3931
Epoch 30/100 — Loss: 0.2112
Epoch 40/100 — Loss: 0.1786
Epoch 50/100 — Loss: 0.1626
Epoch 60/100 — Loss: 0.1474
Epoch 70/100 — Loss: 0.1412
Epoch 80/100 — Loss: 0.1393
Epoch 90/100 — Loss: 0.1372
Epoch 100/100 — Loss: 0.1354


In [19]:
# CELL 7 — Evaluation
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits)   # convert logits -> probability
    test_preds = (test_probs >= 0.5).float()

    accuracy = (test_preds == y_test_t).float().mean().item()
    print(f"Test Accuracy: {accuracy*100:.2f}%")

# Inspect a few example predictions
for i in range(5):
    print(f"Prob spam: {test_probs[i].item():.2f} | Predicted: {int(test_preds[i].item())} | Actual: {int(y_test_t[i].item())}")

Test Accuracy: 94.80%
Prob spam: 0.01 | Predicted: 0 | Actual: 0
Prob spam: 0.04 | Predicted: 0 | Actual: 0
Prob spam: 0.01 | Predicted: 0 | Actual: 0
Prob spam: 0.92 | Predicted: 1 | Actual: 1
Prob spam: 0.00 | Predicted: 0 | Actual: 0
